# **Pruebas Funcionales del TIF**

Como primer paso se importan las librerías necesarias para procesar y graficar señales fisiológicas, y se cargan las clases principales de la librería TIF.py para representar, describir y analizar señales como EEG, ECG y EMG

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import copy
import scipy.signal
from matplotlib.patches import Patch
from scipy.signal import find_peaks, spectrogram
from scipy.signal import hilbert, spectrogram
from TIF import Info, Anotaciones, RawSignal, EEGSignal

## **Clase Info**

> La clase Info permite almacenar información relevante sobre una señal fisiológica, como el nombre del experimentador, los datos del sujeto, los nombres y tipos de canales, los canales defectuosos, la frecuencia de muestreo y una descripción del registro.

En este fragmento se definen los metadatos del registro fisiológico. Se crea una lista con los nombres de los canales (`["F2", "F3"]`) y se indica que ambos son de tipo `"ecg"`. Además, se construye un diccionario con la información del sujeto, que incluye su nombre y edad. Estos datos serán utilizados luego para inicializar un objeto de la clase `Info`.

In [41]:
canales = ["F2", "F3"]
tipos_canales = ["ecg"] * len(canales)
sujeto = {"Nombre": "Juan Acosta",
          "Edad": 26}

En este bloque se crea un objeto `Info` con los datos del experimentador, sujeto, canales y descripción. Luego, se accede a la información almacenada utilizando distintos métodos de la clase

In [42]:
info = Info("Juan Acosta", sujeto, canales, tipos_canales, None, "Pruebita", 512)

Si la cantidad de nombres de canales y tipos de canales no coincide, el objeto `Info` lanza un error al momento de su creación.

In [64]:
canales1=["F10","F50",["13"]]
tipos_canales1=["emg"]
sujeto1={"Nombre":"Flor Sosa", "Edad": 24}

In [67]:
info1 = Info("Flor Sosa", sujeto1, canales1,tipos_canales1, None, "Prueba",201)

ValueError: La cantidad de canales y los tipos de canales deben ser la misma

### ***Método*** ***__ contains __:*** Verifica si una clave está presente en el objeto. Devuelve `True` si la clave existe y `False` en caso contrario.

In [43]:
print(info.__contains__("Experimentador"))

True


In [44]:
print(info.__contains__("tipos_canales"))

False


### ***Método*** ***__ getitem __:*** Obtiene el valor asociado a una clave. Si la clave existe dentro de info devuelve la clave, sino retorna false

In [45]:
print("Nombre del experimentador:",info.__getitem__("Experimentador"))

Nombre del experimentador: Juan Acosta


In [58]:
print(info.__getitem__("tipos_canales"))

False


### ***Método*** ***__ len __:*** Devuelve la cantidad de elementos guardados.

In [46]:
print(info.__len__())

7


### ***Método get:*** Obtiene el valor asociado a una clave específica del objeto. Si la clave existe, devuelve su valor, si no existe, devuelve `False`.

In [60]:
info.get("Frecuencia muestreo")

512

### ***Método keys():*** Devuelve la cantidad de elementos guardados.

In [47]:
print(info.keys()) 

['Experimentador', 'Sujeto', 'Nombre canales', 'Tipo canales', 'Canales malos', 'Descripción', 'Frecuencia muestreo']


### ***Método item():*** Dvuelve una tupla con clave y valor para una clave específica.

In [48]:
print(info.item("Experimentador"))
print(info.item("Nombre canales"))

('Experimentador', 'Juan Acosta')
('Nombre canales', ['F2', 'F3'])


Si al método item() le damos una clave que no existe en el objeto, devuelve simplemente `False`.

In [59]:
print(info.item("Tipo de electrodo"))

False


### ***Método rename_channels:***  Permite renombrar canales de forma segura.

In [49]:
print(info.rename_channels("F2", "F1"))
print(info.item("Nombre canales"))

True
('Nombre canales', ['F1', 'F3'])


Si al `método rename_channels()` se le pasa un valor entero como nombre del canal (por ejemplo, 2 en lugar de "F3"), el método intentará buscar ese número en la lista de nombres de canales, y al no encontrarlo retornara un False.

In [50]:
info.rename_channels(2, 5)
print(info.item("Nombre canales"))

print(info.rename_channels(2,5))

('Nombre canales', ['F1', 'F3'])
False


### ***Método eliminar_elementos:*** Permite eliminar uno o varios elementos de una lista asociada a una clave específica del diccionario interno.

In [51]:
info.eliminar_elementos(key="Nombre canales", elementos=["F3"])
print(info.item("Nombre canales"))

('Nombre canales', ['F1'])


El método `eliminar_elementos` lanza errores si la clave o los elementos a eliminar no existen, lo cual es una forma de protección ante errores silenciosos. Esto se debe a dos validaciones internas

In [55]:
info.eliminar_elementos(key="Canal", elementos=["F1"])

KeyError: "La clave 'Canal' no existe en Info."

In [56]:
info.eliminar_elementos(key="Nombre canales", elementos=["F2"])

ValueError: El elemento 'F2' no se encuentra en la lista asociada a 'Nombre canales'.

## **Clase Anotaciones**

> La clase Anotaciones permite almacenar y gestionar eventos asociados a una señal fisiológica, como estímulos, artefactos o respuestas. Cada evento se define por un tiempo de inicio, una duración y una descripción.

Se definen tres listas que representan las propiedades de los eventos: los tiempos de inicio, la duración y la descripción de cada anotación. Estos datos se usarán para crear un objeto de la clase `Anotaciones`.

In [68]:
inicio = [5.0, 12.5, 20.0]
duracion = [2.0, 3.0, 3.5]
descripcion = ['Inicio_Experimento', 'Evento_1', 'Evento_2']

Se crea un objeto de la clase `Anotaciones` utilizando las listas de inicio, duración y descripción definidas previamente. Esto permite almacenar los eventos en forma estructurada para su posterior uso o análisis.

In [69]:
anotaciones = Anotaciones(onset=inicio, duration=duracion, description=descripcion)

Si se proporciona un archivo (file) como argumento, el constructor intentará cargar las anotaciones desde ese archivo usando el método load(). Si el archivo no existe o su formato no es correcto, se generará un error al momento de leerlo.

In [70]:
anotaciones1= Anotaciones()

ValueError: Debe ingresar las anotaciones manualmente o cargarlas a partir de un archivo

Si las listas `onset` y `duration` no tienen la misma cantidad de elementos, el constructor lanza un `ValueError` para evitar que se creen anotaciones inconsistentes.

In [72]:
inicio1 = [5.0, 12.5, 20.0, 5.2]
duracion1 = [2.0, 3.0, 3.5]
descripcion1 = ['Inicio_Experimento', 'Evento_1', 'Evento_2']

anotaciones2 = Anotaciones(onset= inicio1, duration=duracion, description=descripcion)

ValueError: Onset y duration deben tener la misma cantidad de elementos

### ***Método get_annotations:*** Se utiliza para obtener todas las anotaciones almacenadas en el objeto. El resultado es un DataFrame que muestra, en forma de tabla, los tiempos de inicio, duración y descripción de cada evento.

In [11]:
anotaciones.get_annotations()

,Inicio,Duracion,Descripcion
0,5.0,2.0,Inicio_Experimento
1,12.5,3.0,Evento_1
2,20.0,3.5,Evento_2


### ***Método recorte:*** Permite limpiar las anotaciones, conservando solo aquellas que ocurren dentro de un intervalo de tiempo específico, y eliminando las que están fuera del rango definido por `tmin` y `tmax`.


In [ ]:
anotaciones.recorte(tmin=10, tmax=25)

anotaciones.get_annotations()

,Inicio,Duracion,Descripcion
1,12.5,3.0,Evento_1
2,20.0,3.5,Evento_2


In [14]:
# Agrego una anotación
nueva_anotacion = [3, 4, "Evento 3"]
anotaciones.add(nueva_anotacion)

True

In [15]:
anotaciones.get_annotations()

,Inicio,Duracion,Descripcion
0,5.0,2.0,Inicio_Experimento
1,3.0,4.0,Evento 3


In [16]:
# Elimino una sola anotación en especifica
anotaciones.remove(nueva_anotacion)

True

In [17]:
anotaciones.get_annotations()

,Inicio,Duracion,Descripcion
0,5.0,2.0,Inicio_Experimento


In [18]:
# Busco una anotación (no aparecera nada porque la elimine anteriormente)
anotaciones.find(nueva_anotacion)

,Inicio,Duracion,Descripcion


In [19]:
anotaciones.save("Anotaciones")

In [20]:
anotaciones.load("Anotaciones.csv")

,Unnamed: 0,Inicio,Duracion,Descripcion
0,0,5.0,2.0,Inicio_Experimento


In [21]:
# Genero una instancia anotaciones desde un csv
anotaciones_csv = Anotaciones(file="../2. tests/eeg/eventos_ejemplo.csv")
anotaciones_csv.get_annotations()

,Inicio,Duracion,Descripcion
0,37.548828,5,IZQUIERDA
1,54.048828,5,DERECHA
2,70.482422,5,DERECHA
3,88.632812,5,DERECHA
4,104.984375,5,IZQUIERDA
5,122.300781,5,IZQUIERDA
6,140.085938,5,DERECHA
7,157.287109,5,DERECHA
8,173.853516,5,DERECHA
9,190.685547,5,DERECHA


## **Clase RawSignal**

In [ ]:
eeg_data = np.load("../2. tests/eeg/eeg_signal.npy")

In [ ]:
eeg_data.shape

In [ ]:
# Esto era para ver nomas que habia en el archivo
eeg = pd.DataFrame(eeg_data)
eeg.head()

In [ ]:
# Genero objeto Info

canales = range(1,63)

nombre_canales = []
for canal in canales:
    nombre_canales.append(str(canal))
    
# canales = ["F2", "F3", "F4"]
tipos_canales = ["eeg"] * len(nombre_canales)
sujeto = {"Nombre": "Juan Acosta",
          "Edad": 26}

infoo = Info(experimenter="Juan Acosta", subject_info=sujeto, ch_names=nombre_canales, 
        ch_types=tipos_canales, bads=None, description="Pruebita", fm=512)

print("Claves:", infoo.keys())
print("Canales iniciales:", infoo.data["Nombre canales"])

In [ ]:
# Genero objeto Anotaciones, mediante anotaciones "manuales"

inicio = [5.0, 12.5, 20.0]
duracion = [2.0, 3.0, 3.5]
descripcion = ['Inicio_Experimento', 'Evento_1', 'Evento_2']
anotacioness = Anotaciones(onset=inicio, duration=duracion, description=descripcion)

In [ ]:
# Generar el objeto RawSignal
raw_signal = RawSignal(data=eeg_data, sfreq=512, info=infoo, anotaciones=anotacioness) 
raw_signal.data[1]

In [ ]:
# Método __getitem__() pasando el nombre de un solo canal
raw_signal["3"]

In [ ]:
# Método __getitem__() pasando una lista de canales

raw_signal[["3","4"]]

In [ ]:
# Método __getitem__() pasando un slice con las muestras

raw_signal[512:1024].shape
raw_signal[512:1024]

In [ ]:
# Método __getitem__() pasando una lista de canales y un slice para las muestras

raw_signal[(["1","2"], slice(512, 1024))].shape

In [ ]:
# Obtener muestas con get_data()

# muestras = raw_signal.get_data(start=0, stop=1)   # Sin pasar los canales los selecciona a todos
# muestras, vector = raw_signal.get_data(picks=[0,1,2], start=0, stop=10, times=True, reject=30000)
# print(muestras.shape)
# print(vector)

muestras = raw_signal.get_data(picks=["1","2", "3"], start=0, stop=20)  
print(muestras.shape)

In [ ]:
# Elimina el canal "F2"
nueva_rawsignal = raw_signal.drop_channels(["1"])

In [ ]:
# Verifico que la nueva instancia de RawSignal elimino un canal
print("Canales restantes:", nueva_rawsignal.info["Nombre canales"])

In [ ]:
datos = nueva_rawsignal.describe("Datos.csv")
datos

In [ ]:
nueva_rawsignal.data.shape

In [ ]:
# Uso la nueva instancia RawSignal y selecciono dos canales 
print(nueva_rawsignal.get_data(picks=["3", "4"], start=0, stop=10).shape)

In [ ]:
# Recortar la señal (tiempo). Método crop() desde el primer objeto Rawsignal

rawsignal_recortada = raw_signal.crop(tmin= 5, tmax=50)   # Recorte
datos, tiempo = rawsignal_recortada.get_data(picks=["2", "3"], start=0, stop=5, times=True)
print(datos.shape)
print("Vector temporal:", tiempo)

In [ ]:
# Metodo describe() con la primar instancia de RawSignal que tiene el Objeto Info

raw_signal = RawSignal(data=eeg_data, sfreq=512, info=infoo, anotaciones=anotacioness)
# muestras = raw_signal_2.get_data(picks=["A2", "F3"], start=0, stop=100)
datos = raw_signal.describe("Datos.csv")
datos

In [ ]:
# Metodo describe() sin el Objeto Info. Si no le paso este objeto veo todos los canales y el Tipo de canal se establece en desconocido

raw_signal_sinInfo = RawSignal(data=eeg_data, sfreq=512, anotaciones=anotacioness)
datos2 = raw_signal_sinInfo.describe("Datos2.csv")
datos2

In [ ]:
# Metodo pick()

canales = range(1,63)

nombre_canales = []
for canal in canales:
    nombre_canales.append(str(canal))
    
tipos_canales = ["eeg"] * len(nombre_canales)
sujeto = {"Nombre": "Juan Acosta",
          "Edad": 26}

info2 = Info(experimenter="Juan Acosta", subject_info=sujeto, ch_names=nombre_canales,
            ch_types=tipos_canales, bads=None, description="Pruebita", fm=512)

print("Canales iniciales:", info2.data["Nombre canales"])
print()

raw_signal_2 = RawSignal(data=eeg_data, sfreq=512, info=info2, anotaciones=anotacioness)

# Aca se genera el subset con el metodo pick()
subset = raw_signal_2.pick(canales=["2", "62"])            # Le pasamos los canales que queremos en una lista (canales)
print("Canales del subset:", subset.info["Nombre canales"])
datos = subset.describe("Datoss.csv")                       # Usamos describe para ver algunos datos del subset
datos

In [ ]:
# Metodo pick(). Sin el objeto Info (toma los 62 canales) y usando "slice" selecciona algunos

raw_signal_2 = RawSignal(data=eeg_data, sfreq=512, anotaciones=anotacioness)

subset = raw_signal_2.pick(slice=[0,6])       # Obtenemos los canales de 0 a 4. (El 4 no se incluye)
# subset = raw_signal_2.pick(canales=[0,2,4])
asd = subset.get_data()
asd.shape
datos = subset.describe("Datoss.csv")   # Vemos algunos datos de los canales seleccionados 
datos

In [ ]:
# Generamos un nuevo onjeto Anotaciones. Esta vez se cargan desde un archivo csv
anotaciones_file = Anotaciones(file="../2. tests/eeg/eventos_ejemplo.csv")
anotaciones_file.get_annotations().head()

In [ ]:
# Cambiamos las anotaciones RawSignal
raw_signal_2.set_anotaciones(anotaciones_file)

In [ ]:
# Instanciamos otro objeto RawSignal. Le pasamos Info y las Anotaciones que cargamos del archivo
signal = RawSignal(data=eeg_data, sfreq=512, info=infoo, anotaciones=anotaciones_file)

# Vemos algunas de las anotaciones
signal.anotaciones.get_annotations().head()

In [ ]:
# Recortamos "signal"
recorte_señal = signal.crop(tmin=35, tmax=60)
# recorte_señal.info.get("Nombre canales")

In [ ]:
# Vemos el vector de tiempo para verificar que recortamos la señal 
d, t = recorte_señal.get_data(picks=["2"], times=True)
t

In [ ]:
# Verificamos que tambien se recortaron las anotaciones 
recorte_señal.anotaciones.get_annotations()

In [ ]:
print("first_samp:", recorte_señal.first_samp)
print("sfreq:", recorte_señal.sfreq)
print("Duración de la señal :", recorte_señal.data.shape[1] / recorte_señal.sfreq)

In [ ]:
# Gráficamos nuestra señal recortada
recorte_señal.plot(picks=["2"], show_anotaciones=True)

In [ ]:
# Filtramos la señal
filtrada = recorte_señal.filter(l_freq=20, h_freq=100)

In [ ]:
# Graficarmos la señal filtrada
filtrada.plot(picks=["2"])

## **Clase EEGSignal**

In [ ]:
# Generamos el objeto Info
info_eeg = Info(experimenter= "Juan", subject_info=sujeto, ch_names=nombre_canales, ch_types=tipos_canales,
                bads=None, description= "Prueba", fm= 512 )

anotaciones_file = Anotaciones(file="../2. tests/eeg/eventos_ejemplo.csv")

eeg_data = np.load("../2. tests/eeg/eeg_signal.npy")

In [ ]:
# Instanciar clase EEG

eeg = EEGSignal(data=eeg_data, sfreq=512, info=info_eeg, anotaciones=anotaciones_file, first_samp=0, referencia="canal", canal="1")

In [ ]:
# Le cambio la referencia a un canal
ref = eeg.set_reference(reference="canal", channel="2")
ref.shape
# ref

In [ ]:
ref_promedio = eeg.set_reference(reference="promedio")
ref_promedio.shape
# ref_promedio

In [ ]:
frec, espectro = eeg.espectro_frecuencias(picks=["1","5"],plot=True, fmin=0, fmax=60)

In [ ]:
eeg_filtrada = eeg.filter(l_freq=0.1, h_freq=40)

In [ ]:
eeg_filtrada_recortada = eeg_filtrada.crop(tmin=20, tmax=21)

In [ ]:
eeg_filtrada_recortada.plot(picks=["3"])

In [ ]:
e = eeg_filtrada_recortada.plot_hilbert_transform(picks=["3"])

## **Clase ECGSignal**

## **Clase EMGSignal**

In [ ]:
# Cargar datos
emg_data = np.load("../4. tests/emg/emg.npy")  

In [ ]:
emg_data.shape

In [ ]:
emg = EMGSignal(emg_data, sfreq=1000, umbral_microv=2000, start_time=0, end_time=15)

emg.plot_activaciones(canal=0)
emg.plot_spectrogram(canal=0)
emg.plot_hilbert(canal=0)